# Hafta 10 — İnce Ayar, PEFT ve Hizalama

Önceden eğitilmiş bir model, verimli biçimde ve insan tercihleriyle hizalı olarak nasıl işe yarar bir şeye dönüştürülür? Tam ince ayarı, LoRA'yı, talimat ayarını, RLHF'yi ve DPO'yu ele alıyoruz. Uygulamalar baştan sona okunabilecek kadar küçüktür.

## Öğrenme Hedefleri

- Küçük bir Transformer'a bir sınıflandırma görevinde ince ayar yapmak ve felaket boyutunda unutmayı (catastrophic forgetting) ölçmek.
- Düşük Ranklı Uyarlamayı (LoRA) türetmek, sıfırdan bir modül olarak uygulamak ve aynı görevde tam ince ayarla karşılaştırmak.
- Talimat ayarı için veri hattını kurmak (eğitim döngüsü 4. haftadakiyle aynıdır).
- RLHF hedefini ve DPO kaybını türetmek; DPO'yu oyuncak bir politika üzerinde uçtan uca eğitmek.

## Zorunlu Okumalar

- Hu, E. vd. (2021). *LoRA: Low-Rank Adaptation of Large Language Models*.
- Ouyang, L. vd. (2022). *Training Language Models to Follow Instructions with Human Feedback*.
- Rafailov, R. vd. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*.
- Christiano, P. vd. (2017). *Deep Reinforcement Learning from Human Preferences*.

In [ ]:
import sys, math, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
torch.manual_seed(0); np.random.seed(0); random.seed(0)

## 1. Tam ince ayar taban çizgisi

Önceden eğitilmiş ağırlıklardan başlayan standart denetimli öğrenme. Her parametre güncellenir. İki pratik kaygı:

- **Felaket boyutunda unutma.** İnce ayar verisi dar veya öğrenme oranı fazla yüksekse model ön eğitim bilgisini unutur.
- **Depolama.** Her alt görev için ayrı bir ince ayarlı kopya israftır (her kopya modelin tamamıdır).

Bu ödünleşimlerin görünür olduğu küçük, sentetik bir ortam kuruyoruz:

- **Ön eğitim görevi:** her konumda $y_t = (x_t + x_{t-1}) \bmod 30$'u tahmin et. Bu, modelin komşu tokenlara dikkat etmeyi öğrenmesini gerektirir. Düzgün tahminin kaybı $\ln 30 \approx 3{,}40$'tır.
- **İnce ayar görevi:** ilk token son tokendan büyük mü? (2 sınıf). Bu da dizinin iki ucuna dikkat etmeyi gerektirir.

Unutmayı ölçmek için ince ayardan önce ve sonra gövdenin **ön eğitim kaybına** bakıyoruz.

In [ ]:
class TinyTransformer(nn.Module):
    '''İnce ayar deneyleri için asgari, önceden eğitilmiş tarzda bir kodlayıcı.'''
    def __init__(self, V, d=32, h=4, n_layers=2, max_len=16):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.pos = nn.Embedding(max_len, d)
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(
            d_model=d, nhead=h, dim_feedforward=4*d, batch_first=True, dropout=0.0
        ) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, V)

    def encode(self, x):
        T = x.size(1)
        h = self.emb(x) + self.pos(torch.arange(T, device=x.device))
        for layer in self.layers:
            h = layer(h)
        return self.ln(h)

    def forward(self, x):
        return self.lm_head(self.encode(x))

# Modelin rastgele olmaması için oyuncak bir hedefle ön eğitim yap.
V, MAX = 30, 12

def make_pretrain_batch(n=64):
    x = torch.randint(0, V, (n, MAX))
    y = (x + torch.roll(x, 1, dims=1)) % V   # y_t = (x_t + x_{t-1}) mod V
    y[:, 0] = x[:, 0]                        # ilk konumda öncül yok: kendisini kopyala
    return x, y

@torch.no_grad()
def pretrain_loss(model, n=512):
    x, y = make_pretrain_batch(n)
    return F.cross_entropy(model(x).reshape(-1, V), y.reshape(-1)).item()

pretrained = TinyTransformer(V, d=32, h=4, n_layers=2, max_len=MAX)
opt = torch.optim.Adam(pretrained.parameters(), lr=3e-3)
for step in range(1200):
    x, y = make_pretrain_batch()
    loss = F.cross_entropy(pretrained(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
PRETRAIN_LOSS = pretrain_loss(pretrained)
print(f"Ön eğitim kaybı: {PRETRAIN_LOSS:.3f}   (düzgün tahmin: ln {V} = {math.log(V):.3f})")

In [ ]:
# İnce ayar görevi: ilk token son tokendan büyük mü?
def make_clf_data(N, max_len=MAX, vocab=V):
    x = torch.randint(0, vocab, (N, max_len))
    y = (x[:, 0] > x[:, -1]).long()
    return x, y

class FineTuneHead(nn.Module):
    def __init__(self, backbone, d, n_classes=2):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(d, n_classes)
    def forward(self, x):
        h = self.backbone.encode(x).mean(dim=1)  # ortalama havuzlama
        return self.classifier(h)

def count_trainable(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

@torch.no_grad()
def val_acc(model):
    return (model(X_va).argmax(-1) == y_va).float().mean().item()

def finetune(model, lr, steps=400):
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    curve = []
    for step in range(steps):
        idx = torch.randint(0, len(X_tr), (64,))
        loss = F.cross_entropy(model(X_tr[idx]), y_tr[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 20 == 0:
            curve.append((step, loss.item(), val_acc(model)))
    return curve

import copy
X_tr, y_tr = make_clf_data(4000)
X_va, y_va = make_clf_data(1000)

# (a) Tam ince ayar: her parametre güncellenir.
model_full = FineTuneHead(copy.deepcopy(pretrained), d=32).train()
curve_full = finetune(model_full, lr=1e-3)

# (b) Taban çizgisi: gövde dondurulur, yalnızca sınıflandırıcı eğitilir.
model_frozen = FineTuneHead(copy.deepcopy(pretrained), d=32)
for p in model_frozen.backbone.parameters():
    p.requires_grad = False
curve_frozen = finetune(model_frozen, lr=1e-2)

print(f"Tam ince ayar      : doğrulama doğruluğu = {val_acc(model_full):.3f}   eğitilen parametre = {count_trainable(model_full):,}")
print(f"Donmuş gövde       : doğrulama doğruluğu = {val_acc(model_frozen):.3f}   eğitilen parametre = {count_trainable(model_frozen):,}")
print(f"\nFelaket boyutunda unutma — gövdenin ön eğitim kaybı:")
print(f"  ince ayardan önce : {PRETRAIN_LOSS:.3f}")
print(f"  tam ince ayardan sonra: {pretrain_loss(model_full.backbone):.3f}   (düzgün tahmin = {math.log(V):.3f})")

## 2. LoRA — Düşük Ranklı Uyarlama

**Fikir (Hu vd., 2021).** Önceden eğitilmiş ağırlıkları $W_0$ dondur. Düşük ranklı bir düzeltme ekle:

$$W = W_0 + \Delta W = W_0 + B A, \quad B \in \mathbb{R}^{d \times r}, \ A \in \mathbb{R}^{r \times k}, \ r \ll \min(d, k).$$

$A \sim \mathcal{N}(0, \sigma^2)$ ve $B = 0$ ile başlatırız; böylece model önceden eğitilmiş olanla birebir aynı başlar. Yalnızca $A$ ve $B$ eğitilir — büyük modellerde genellikle orijinal parametre sayısının %1'inden azı — ve alt görev performansı neredeyse eşdeğerdir.

Öğrenme oranının en uygun değeri $r$'ye bağlı olmasın diye bir $\alpha / r$ ölçek çarpanı uygulanır:

$$h = W_0 x + \frac{\alpha}{r} B A x.$$

Aşağıda önce LoRA'yı genel bir `nn.Linear` sarmalayıcısı olarak uyguluyor, ardından matematiği tamamen görünür kılmak için LoRA'nın gömülü olduğu küçük bir öz-dikkat katmanı yazıyoruz. Son olarak `LoRALinear`'ı 1. bölümdeki modelin ileri besleme katmanlarına uygulayıp aynı görevde eğitiyoruz.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base_linear.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base_linear.out_features, r))
        self.scaling = alpha / r
        self.enabled = True   # False olduğunda katman tam olarak orijinal hâline döner

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + (x @ self.A.T @ self.B.T) * self.scaling
        return out

# Not: nn.TransformerEncoderLayer, Q, K ve V için tek bir birleşik in_proj_weight tensörü kullanır;
# bu yüzden dikkat projeksiyonlarına LoRA eklemek PyTorch'un iç yapısına müdahale gerektirir.
# Üretimde bunun için `peft` kütüphanesi kullanılır. Burada iki yol izliyoruz:
#  (1) Matematiği görmek için LoRA'nın gömülü olduğu küçük bir öz-dikkat katmanı yazmak,
#  (2) Eğitim deneyi için LoRA'yı ileri besleme katmanlarına (linear1 / linear2) uygulamak.

class LoRASelfAttention(nn.Module):
    def __init__(self, d, h, r=4, alpha=8):
        super().__init__()
        self.h, self.dk = h, d // h
        # Temel (dondurulmuş) Q, K, V projeksiyonları.
        self.W_q = nn.Linear(d, d); self.W_k = nn.Linear(d, d); self.W_v = nn.Linear(d, d)
        self.W_o = nn.Linear(d, d)
        # LoRA yalnızca Q ve V üzerinde (en yaygın seçim).
        self.A_q = nn.Parameter(torch.randn(r, d) * 0.01); self.B_q = nn.Parameter(torch.zeros(d, r))
        self.A_v = nn.Parameter(torch.randn(r, d) * 0.01); self.B_v = nn.Parameter(torch.zeros(d, r))
        self.scaling = alpha / r
        for p in [self.W_q.weight, self.W_q.bias, self.W_k.weight, self.W_k.bias,
                  self.W_v.weight, self.W_v.bias, self.W_o.weight, self.W_o.bias]:
            p.requires_grad = False

    def forward(self, x):
        B, T, D = x.shape
        q = self.W_q(x) + (x @ self.A_q.T @ self.B_q.T) * self.scaling
        k = self.W_k(x)
        v = self.W_v(x) + (x @ self.A_v.T @ self.B_v.T) * self.scaling
        q = q.view(B, T, self.h, self.dk).transpose(1, 2)
        k = k.view(B, T, self.h, self.dk).transpose(1, 2)
        v = v.view(B, T, self.h, self.dk).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)
        out = F.softmax(scores, dim=-1) @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

# Parametre tasarrufunu göster.
base_attn = LoRASelfAttention(d=64, h=4, r=4, alpha=8)
full_attn_params = sum(p.numel() for p in base_attn.parameters() if not p.requires_grad)
print(f"Dikkat bloğunun dondurulmuş parametreleri: {full_attn_params:,}")
print(f"Dikkat bloğunun LoRA ile eğitilen parametreleri: {count_trainable(base_attn):,}")
print("  -- yalnızca LoRA A/B matrisleri eğitilir; W_q, W_k, W_v, W_o dondurulmuş kalır.")

# 1. bölümdeki görevde LoRA ile ince ayar: gövde dondurulur, ileri besleme katmanlarına LoRA eklenir.
model_lora = FineTuneHead(copy.deepcopy(pretrained), d=32)
for p in model_lora.backbone.parameters():
    p.requires_grad = False
for layer in model_lora.backbone.layers:
    layer.linear1 = LoRALinear(layer.linear1, r=8, alpha=16)
    layer.linear2 = LoRALinear(layer.linear2, r=8, alpha=16)
curve_lora = finetune(model_lora, lr=5e-3, steps=800)

print(f"\n{'yöntem':<16} {'doğruluk':>9} {'eğitilen parametre':>20}")
for name, m in [("tam ince ayar", model_full), ("LoRA (r=8)", model_lora), ("donmuş gövde", model_frozen)]:
    print(f"{name:<16} {val_acc(m):>9.3f} {count_trainable(m):>20,}")

# LoRA'nın unutmaya karşı güvencesi: adaptörler kapatıldığında gövde birebir önceden eğitilmiş modeldir.
for layer in model_lora.backbone.layers:
    layer.linear1.enabled = layer.linear2.enabled = False
print(f"\nAdaptörler kapalıyken gövdenin ön eğitim kaybı: {pretrain_loss(model_lora.backbone):.3f}"
      f"  (ince ayar öncesi: {PRETRAIN_LOSS:.3f})")
for layer in model_lora.backbone.layers:
    layer.linear1.enabled = layer.linear2.enabled = True

plt.figure(figsize=(8, 4))
for name, curve in [("tam ince ayar", curve_full), ("LoRA (r=8)", curve_lora), ("donmuş gövde", curve_frozen)]:
    plt.plot([s for s, _, _ in curve], [a for _, _, a in curve], label=name)
plt.xlabel('adım'); plt.ylabel('doğrulama doğruluğu'); plt.title('İnce ayar yöntemlerinin karşılaştırması')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 3. Talimat ayarı

Her görevi `(talimat, girdi, çıktı)` üçlüleri olarak çerçeveleyin ve dil modeline `<talimat><girdi><çıktı>` birleşimi üzerinde, kayıp yalnızca çıktı tokenlarında olacak şekilde ince ayar yapın. Model, ön eğitim sırasında hiçbir görevi uçtan uca görmeden talimatları izlemeyi öğrenir.

Aşağıda veri hattının taslağını çiziyoruz. Gerçek bir derlemle (Alpaca, OpenAssistant, FLAN) aynı kod, bir modeli birçok görevde talimat izleyecek şekilde eğitir; eğitim döngüsünün kendisi 4. haftadaki nedensel dil modeli döngüsüyle aynıdır.

In [ ]:
# Veri hattı taslağı — oyuncak sözlük üzerinde uçtan uca çalıştırılabilir.
INSTRUCTIONS = [
    ("ters çevir:",  lambda x: x[::-1]),
    ("sırala:",      lambda x: sorted(x)),
    ("ikile:",       lambda x: [t for t in x for _ in range(2)]),
    ("ilk yarı:",    lambda x: x[:len(x)//2]),
]

def make_instruction_example(vocab_size=20, max_len=4):
    instr, fn = random.choice(INSTRUCTIONS)
    inp = [random.randint(0, vocab_size - 1) for _ in range(max_len)]
    out = fn(inp)
    return instr, inp, out

for _ in range(4):
    instr, inp, out = make_instruction_example()
    print(f"{instr:>14s} girdi={inp}  ->  çıktı={out}")

print("\nTalimat ayarı için şunları birleştiririz: <BOS> talimat <SEP> girdi <SEP> çıktı <EOS>")
print("ve kaybı yalnızca ikinci <SEP>'ten sonraki tokenlarda hesaplarız.")

## 4. RLHF — standart hat

Üç aşama (Ouyang vd., 2022):

1. Gösterimler üzerinde **denetimli ince ayar (SFT)**. Bunun veri hattını §3'te kurduk.
2. **Ödül modellemesi.** $x$ istemi verildiğinde bir insanın $y^+$'yı $y^-$'ya tercih ettiği $(y^+, y^-)$ çiftlerini toplayın. **Bradley–Terry** kaybıyla skaler bir ödül modeli $r_\phi(x, y)$ eğitin:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y^+, y^-)} \log \sigma(r_\phi(x, y^+) - r_\phi(x, y^-)).$$

3. SFT referansına karşı bir KL cezasıyla **ödül modeline karşı PPO**:

$$\max_{\pi_\theta} \mathbb{E}_{y \sim \pi_\theta} [r_\phi(x, y)] - \beta \, D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}}).$$

PPO mekanik olarak ağırdır. Sonraki bölüm, onu tamamen ortadan kaldıran DPO'yu sunuyor.

## 5. Doğrudan Tercih Optimizasyonu (DPO)

Rafailov vd. (2023) şunu gösterdi: yukarıdaki KL düzenlileştirmeli RL probleminin kapalı biçimli bir çözümü vardır:

$$\pi^*(y \mid x) \propto \pi_{\text{ref}}(y \mid x) \exp\!\left(\frac{r(x, y)}{\beta}\right),$$

bu da $r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$ demektir. Bunu Bradley–Terry kaybına koymak $r$'yi (ve $Z(x)$'i) yok eder ve doğrudan $\pi_\theta$ üzerinde denetimli bir kayıp verir:

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y^+, y^-)} \log \sigma\!\left( \beta \log \frac{\pi_\theta(y^+ \mid x)}{\pi_{\text{ref}}(y^+ \mid x)} - \beta \log \frac{\pi_\theta(y^- \mid x)}{\pi_{\text{ref}}(y^- \mid x)} \right).$$

Ödül modeli yok, PPO yok, açılım (rollout) yok. Yalnızca tercih çiftleri üzerinde bir sınıflandırma kaybı.

Aşağıda önce kaybı uyguluyor ve politika referansla aynıyken kaybın tam olarak $\log 2$ olduğunu doğruluyoruz (o anda örtük ödül marjı sıfırdır). Ardından DPO'yu oyuncak bir politika üzerinde uçtan uca eğitiyoruz.

In [ ]:
def dpo_loss(policy_logits_pos, policy_logits_neg,
             ref_logits_pos, ref_logits_neg,
             labels_pos, labels_neg, beta=0.1):
    '''Bir tercih çifti yığını için DPO kaybını hesaplar.

    policy_logits_*: (B, T, V) eğitilebilir modelden.
    ref_logits_*   : (B, T, V) dondurulmuş referans (SFT) modelden.
    labels_*       : (B, T) — hedef tokenlar; -100 yok sayılacak tokenları (ör. istem) işaretler.
    '''
    def seq_logprob(logits, labels):
        # Etiketli tokenların log-olasılıklarını topla.
        logp = F.log_softmax(logits, dim=-1)
        mask = (labels != -100)
        safe_labels = labels.clamp(min=0)
        tok_logp = logp.gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
        return (tok_logp * mask).sum(-1)

    pol_pos = seq_logprob(policy_logits_pos, labels_pos)
    pol_neg = seq_logprob(policy_logits_neg, labels_neg)
    ref_pos = seq_logprob(ref_logits_pos,    labels_pos)
    ref_neg = seq_logprob(ref_logits_neg,    labels_neg)

    logits = beta * ((pol_pos - ref_pos) - (pol_neg - ref_neg))
    return -F.logsigmoid(logits).mean(), logits.mean().item()

# Duman testi: politika referansla AYNIYKEN örtük ödül marjı 0, kayıp tam olarak log 2'dir.
B, T, V_local = 4, 8, 25
same_pos, same_neg = torch.randn(B, T, V_local), torch.randn(B, T, V_local)
loss, margin = dpo_loss(
    same_pos, same_neg, same_pos.clone(), same_neg.clone(),
    torch.randint(0, V_local, (B, T)), torch.randint(0, V_local, (B, T)),
    beta=0.1,
)
print(f"Politika = referans iken DPO kaybı: {loss.item():.4f}  (log 2 = {math.log(2):.4f})")
print(f"Ortalama tercih marjı             : {margin:+.4f}")

# Uçtan uca oyuncak DPO: 8 istem, her biri için 5 olası tek tokenlık yanıt.
# Politika ve referans, istem başına bir logit tablosudur; tercihler her istemde
# "iyi" yanıtı (indeks 0) diğerlerinden birine tercih eder.
N_PROMPTS, N_RESP, BETA = 8, 5, 0.1
torch.manual_seed(0)
ref_table = torch.randn(N_PROMPTS, N_RESP)               # dondurulmuş referans (SFT) politikası
policy_table = nn.Parameter(ref_table.clone())            # politika referanstan başlar
opt_dpo = torch.optim.Adam([policy_table], lr=0.05)

def p_good():
    return F.softmax(policy_table, dim=-1)[:, 0].mean().item()

def kl_to_ref():
    logp, logq = F.log_softmax(policy_table, -1), F.log_softmax(ref_table, -1)
    return (logp.exp() * (logp - logq)).sum(-1).mean().item()

print(f"\nBaşlangıçta ortalama P(iyi yanıt) = {p_good():.3f}   KL(politika || referans) = {kl_to_ref():.3f}")
for step in range(300):
    prompts = torch.randint(0, N_PROMPTS, (32,))
    chosen = torch.zeros(32, dtype=torch.long)            # tercih edilen: indeks 0
    rejected = torch.randint(1, N_RESP, (32,))            # reddedilen: diğerlerinden biri
    pol = policy_table[prompts].unsqueeze(1)              # (B, T=1, V)
    ref = ref_table[prompts].unsqueeze(1)
    loss, margin = dpo_loss(pol, pol, ref, ref, chosen.unsqueeze(1), rejected.unsqueeze(1), beta=BETA)
    opt_dpo.zero_grad(); loss.backward(); opt_dpo.step()
    if (step + 1) % 100 == 0:
        print(f"adım {step+1:3d}  DPO kaybı = {loss.item():.3f}  ortalama P(iyi) = {p_good():.3f}  "
              f"KL(politika || referans) = {kl_to_ref():.3f}")

## 6. Ne zaman DPO, ne zaman RLHF?

| Durum | Tercih edilen |
|---------|-----------|
| Tek bir yüksek kaliteli tercih veri kümesi | **DPO** — daha basit, daha az hareketli parça |
| Çevrimiçi örnekleme, keşif, çok aşamalı optimizasyon | **PPO/RLHF** — açılımları ve açık ödül biçimlendirmeyi destekler |
| Çok büyük tercih veri kümeleri; bir ödül modelini birçok politikaya amorti etmek | **RLHF** — RM'yi bir kez eğit, yeniden kullan |
| Sınırlı mühendislik kapasitesi | **DPO** |

Büyüyen bir literatür (IPO, KTO, ORPO, SimPO) DPO'yu daha da iyileştiriyor. Kavramsal çekirdek aynı kalıyor: tercihler ile politika log-oranları arasında kapalı biçimli bir dönüşüm.

## 7. Alıştırmalar

1. **LoRA rank ablasyonu.** Yukarıdaki sınıflandırma görevinde $r \in \{1, 2, 4, 8, 16, 32\}$ ile LoRA eğitin. Nihai doğruluğu eğitilen parametre sayısına göre çizdirin. Eğrinin dirseğini belirleyin.
2. **Tam ince ayarda unutma.** Tam ince ayarlı ve LoRA ile ince ayarlı modelleri, adaptörler *açıkken* orijinal ön eğitim hedefi üzerinde değerlendirin. Hangisi daha çok unutuyor? Adaptörler kapalıyken ne oluyor ve bu neden LoRA'nın pratikteki büyük avantajlarından biri?
3. **DPO gradyanı doğrulama.** $\partial \mathcal{L}_{\text{DPO}} / \partial \theta$'yı analitik olarak türetin ve küçük bir modelde sonlu farklarla doğrulayın.
4. **$\beta$'nın rolü.** Oyuncak DPO deneyini $\beta \in \{0.01, 0.1, 1.0\}$ ile tekrarlayın. $P(\text{iyi})$ ve KL(politika ‖ referans) nasıl değişiyor? $\beta$ neden bir "referanstan ne kadar uzaklaşılabilir" düğmesi gibi davranıyor?
5. **Ödül istismarı (reward hacking).** *"certainly"* sözcüğünü içeren yanıtları tercih eden küçük bir ödül modeli kurun. Birkaç DPO güncellemesi çalıştırın ve politikanın *"certainly"*'yi gelişigüzel üretmeye başladığını doğrulayın. Bunun ödül modellerinin pratikteki sağlamlığı hakkında ne söylediğini tartışın.

---

## Gelecek Hafta

Hafta 11 — Modern LLM uygulamaları: RAG, araç kullanımı, değerlendirme.